# Lab 01 — Python Essentials for RAG Engineering

**Knowledge base:** `knowledge_base/01_rag_fundamentals/03_rag_architecture.md`

**Concepts:** Lists · Dicts · f-strings · Functions · Type hints

Every pattern here appears directly in the RAG code you write in later labs.

---

In [1]:
import sys
print(f"Python version: {sys.version}")
assert sys.version_info >= (3, 11), "Activate rag-env first:  conda activate rag-env"
print("✅ Environment OK")

Python version: 3.11.15 | packaged by Anaconda, Inc. | (main, Mar 11 2026, 17:12:15) [MSC v.1942 64 bit (AMD64)]
✅ Environment OK


---
## 1 — Lists

In RAG, everything moves through the pipeline as a list:
- Retrieved documents → `List[str]`
- Conversation history → `List[Dict]`
- Similarity scores → `List[float]`

### 1.1 Basic operations

In [2]:
# A retriever returns a list of document strings
documents = [
    "RAG stands for Retrieval Augmented Generation.",
    "LLMs have a training cutoff date and cannot access private data.",
    "Vector databases store embeddings for fast similarity search.",
    "Chunking splits long documents into smaller pieces before embedding.",
]

print(f"Total documents: {len(documents)}")
print(f"First (index 0):  {documents[0]}")
print(f"Last  (index -1): {documents[-1]}")

Total documents: 4
First (index 0):  RAG stands for Retrieval Augmented Generation.
Last  (index -1): Chunking splits long documents into smaller pieces before embedding.


In [3]:
# Slicing — a retriever returns TOP-K results, not all documents
top_2 = documents[:2]
print(f"Top-2 retrieved:\n  {top_2}")

# append vs extend — used when building conversation history
more_docs = ["The retriever scores documents by relevance.",
             "The generator reads retrieved docs and produces an answer."]

documents.extend(more_docs)   # adds EACH item from the list
print(f"\nAfter extend: {len(documents)} documents")

documents.append("One more concept.")   # adds ONE item
print(f"After append:  {len(documents)} documents")

Top-2 retrieved:
  ['RAG stands for Retrieval Augmented Generation.', 'LLMs have a training cutoff date and cannot access private data.']

After extend: 6 documents
After append:  7 documents


### 1.2 List comprehensions

List comprehensions build a new list by transforming or filtering an existing one.
You will use them constantly — especially to label ranked results and filter by score.

In [4]:
# Add a rank label to each document — retrievers return ranked results
labeled = [f"[Rank {i+1}] {doc}" for i, doc in enumerate(documents[:3])]
for doc in labeled:
    print(doc)

[Rank 1] RAG stands for Retrieval Augmented Generation.
[Rank 2] LLMs have a training cutoff date and cannot access private data.
[Rank 3] Vector databases store embeddings for fast similarity search.


In [5]:
# Filter — keep only documents mentioning 'RAG'
# This is the Python equivalent of a keyword metadata filter
rag_docs = [doc for doc in documents if "RAG" in doc]
print(f"Documents about RAG ({len(rag_docs)} found):")
for d in rag_docs:
    print(f"  • {d}")

# The for-loop version — same result, more code
rag_docs_loop = []
for doc in documents:
    if "RAG" in doc:
        rag_docs_loop.append(doc)

print(f"\nBoth produce the same result: {rag_docs == rag_docs_loop}")

Documents about RAG (1 found):
  • RAG stands for Retrieval Augmented Generation.

Both produce the same result: True


---
## 2 — Dictionaries

Every LLM message is a dict: `{'role': ..., 'content': ...}`.
Every retrieved document with metadata is a dict.
The LLM's response is a dict.

### 2.1 Basic operations

In [6]:
# A single LLM message — the exact format utils.py sends to the model
message = {
    "role": "user",
    "content": "What is RAG?"
}

print(f"Role:    {message['role']}")
print(f"Content: {message['content']}")

# .get() — safe access that returns a default instead of raising KeyError
role    = message.get("role", "unknown")
tokens  = message.get("tokens", 0)   # key doesn't exist → returns 0
print(f"\nrole:   {role}")
print(f"tokens: {tokens}  ← key missing, used default")

Role:    user
Content: What is RAG?

role:   user
tokens: 0  ← key missing, used default


In [7]:
# Retrieved documents as a list of dicts — the standard shape from a retriever
retrieved_docs = [
    {"id": 1, "score": 0.95, "text": "RAG retrieves relevant documents before generating."},
    {"id": 2, "score": 0.88, "text": "The retriever searches a vector database."},
    {"id": 3, "score": 0.74, "text": "The LLM reads retrieved docs and generates an answer."},
]

# .items() lets you iterate over both key and value
print("First retrieved document:")
for key, value in retrieved_docs[0].items():
    print(f"  {key}: {value}")

First retrieved document:
  id: 1
  score: 0.95
  text: RAG retrieves relevant documents before generating.


---
## 3 — f-strings and prompt templates

Every RAG prompt is built with an f-string.
The template never changes — only the query and the retrieved documents change each run.

### 3.1 f-string basics

In [8]:
question = "What is retrieval augmented generation?"
context  = "RAG is a technique that improves LLM accuracy by retrieving relevant documents."

# Triple-quoted f-string — this IS the simplest possible RAG prompt
prompt = f"""Answer the question using only the context below.

Context:
{context}

Question:
{question}

Answer:"""

print(prompt)

Answer the question using only the context below.

Context:
RAG is a technique that improves LLM accuracy by retrieving relevant documents.

Question:
What is retrieval augmented generation?

Answer:


### 3.2 Building a context block from a list of dicts

The retriever returns a list of dicts. You flatten them into a single text block to inject into the prompt.

In [9]:
# List comprehension + join — the standard context-block pattern
context_lines = [
    f"[Doc {doc['id']} | score {doc['score']:.2f}] {doc['text']}"
    for doc in retrieved_docs
]

context_block = "\n".join(context_lines)
print(context_block)

[Doc 1 | score 0.95] RAG retrieves relevant documents before generating.
[Doc 2 | score 0.88] The retriever searches a vector database.
[Doc 3 | score 0.74] The LLM reads retrieved docs and generates an answer.


In [10]:
# Full augmented prompt — query + context block
user_question = "How does a RAG system work?"

full_prompt = f"""Answer the question using the retrieved documents below.
Do not use any knowledge outside these documents.

Retrieved Documents:
{context_block}

Question: {user_question}

Answer:"""

print(full_prompt)

Answer the question using the retrieved documents below.
Do not use any knowledge outside these documents.

Retrieved Documents:
[Doc 1 | score 0.95] RAG retrieves relevant documents before generating.
[Doc 2 | score 0.88] The retriever searches a vector database.
[Doc 3 | score 0.74] The LLM reads retrieved docs and generates an answer.

Question: How does a RAG system work?

Answer:


---
## 4 — Functions and type hints

`utils.py` is full of functions with type hints like `List[Dict]`, `str`, `float`.
Type hints do **not** enforce types at runtime — they are documentation for you and your editor.

In [11]:
from typing import List, Dict

def format_documents(docs: List[Dict], max_docs: int = 3) -> str:
    """
    Convert a list of retrieved document dicts into a single formatted string
    ready to inject into a RAG prompt.

    Args:
        docs:     List of dicts, each with 'id', 'score', and 'text' keys.
        max_docs: Maximum number of documents to include. Default 3.

    Returns:
        A formatted string — the context block for the augmented prompt.
    """
    top_docs = docs[:max_docs]
    lines = [f"[Doc {doc['id']} | score {doc['score']:.2f}] {doc['text']}" for doc in top_docs]
    return "\n".join(lines)


context = format_documents(retrieved_docs, max_docs=2)
print(context)

[Doc 1 | score 0.95] RAG retrieves relevant documents before generating.
[Doc 2 | score 0.88] The retriever searches a vector database.


In [12]:
def build_rag_prompt(query: str, docs: List[Dict], max_docs: int = 3) -> str:
    """
    Build a complete augmented prompt from a query and retrieved documents.

    Args:
        query:    The user's question.
        docs:     Retrieved documents (list of dicts with 'id', 'score', 'text').
        max_docs: How many documents to include in the context.

    Returns:
        The full augmented prompt string ready to send to the LLM.
    """
    context = format_documents(docs, max_docs=max_docs)
    return f"""Answer the question using only the context below.

Context:
{context}

Question: {query}

Answer:"""


prompt = build_rag_prompt("How does a RAG system work?", retrieved_docs, max_docs=2)
print(prompt)

Answer the question using only the context below.

Context:
[Doc 1 | score 0.95] RAG retrieves relevant documents before generating.
[Doc 2 | score 0.88] The retriever searches a vector database.

Question: How does a RAG system work?

Answer:


---
## 5 — Exercises

Fill in each `# YOUR CODE HERE` using only what you learned in this lab.

In [13]:
# Exercise 1 — Build a conversation history list with 3 message dicts:
#   Turn 1: user asks "What is an embedding?"
#   Turn 2: assistant replies "An embedding is a numerical representation of text."
#   Turn 3: user asks "Why is that useful?"

conversation = [
    # YOUR CODE HERE
]

assert len(conversation) == 3
assert conversation[0]['role'] == 'user'
assert conversation[1]['role'] == 'assistant'
assert conversation[2]['role'] == 'user'
print("✅ Exercise 1 passed")

AssertionError: 

In [ ]:
# Exercise 2 — Use a list comprehension to keep only documents with score > 0.80

scored_docs = [
    {"id": 1, "score": 0.95, "text": "Embeddings capture semantic meaning."},
    {"id": 2, "score": 0.78, "text": "TF-IDF is a keyword scoring method."},
    {"id": 3, "score": 0.91, "text": "Cosine similarity measures direction in vector space."},
    {"id": 4, "score": 0.65, "text": "BM25 improves on TF-IDF with saturation."},
    {"id": 5, "score": 0.83, "text": "Hybrid search combines keyword and semantic."},
]

high_confidence = # YOUR CODE HERE

print(f"Documents above threshold: {len(high_confidence)}")
for doc in high_confidence:
    print(f"  [score {doc['score']}] {doc['text']}")

assert len(high_confidence) == 3
print("✅ Exercise 2 passed")

In [ ]:
# Exercise 3 — Write a function that takes a query and a list of scored docs
# and returns: "Query: <query>\nRetrieved N documents. Top score: X.XX"

def summarize_retrieval(query: str, docs: List[Dict]) -> str:
    # YOUR CODE HERE
    pass

result = summarize_retrieval("What is RAG?", scored_docs)
print(result)

assert "Query: What is RAG?" in result
assert "5" in result
print("✅ Exercise 3 passed")

---
**Lab 01 complete.**

You now have all the Python patterns used in every lab that follows.

**Next:** `lab02_environment_and_utils.ipynb` — set up your LLM connection and understand `utils.py`.